# AI Thinking vs Human Thinking

Mini Project — Tic-Tac-Toe comparison using Minimax AI and a heuristic human player.

This notebook preserves the program described in the supplied report.

In [ ]:
import time
import random

# ---------------------------------------------------------------------
# Board utilities
# ---------------------------------------------------------------------
EMPTY, AI, HUMAN = " ", "A", "H"

def print_board(board):
    rows = [board[i:i + 3] for i in range(0, 9, 3)]
    for r in rows:
        print(" | ".join(r))
        print("-" * 9)

def winner(board):
    lines = [(0, 1, 2), (3, 4, 5), (6, 7, 8),
             (0, 3, 6), (1, 4, 7), (2, 5, 8),
             (0, 4, 8), (2, 4, 6)]
    for a, b, c in lines:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    if EMPTY not in board:
        return "DRAW"
    return None

def available_moves(board):
    return [i for i, v in enumerate(board) if v == EMPTY]

# ---------------------------------------------------------------------
# AI thinking: Minimax -> exhaustive, logical evaluation of every
# possible future, always converges on the optimal move.
# ---------------------------------------------------------------------
def minimax(board, player):
    result = winner(board)
    if result == AI:
        return 1, None
    if result == HUMAN:
        return -1, None
    if result == "DRAW":
        return 0, None

    moves_scores = []
    for move in available_moves(board):
        board[move] = player
        score, _ = minimax(board, AI if player == HUMAN else HUMAN)
        board[move] = EMPTY
        moves_scores.append((score, move))

    if player == AI:  # maximizing
        best = max(moves_scores, key=lambda x: x[0])
    else:  # minimizing
        best = min(moves_scores, key=lambda x: x[0])
    return best

def ai_move(board):
    _, move = minimax(board, AI)
    return move

# ---------------------------------------------------------------------
# Human thinking: fast heuristic rules (quick pattern recognition),
# no exhaustive search, small chance of a "human" slip.
# ---------------------------------------------------------------------
def human_move(board):
    moves = available_moves(board)

    # Heuristic 1: take a winning move if obvious at a glance
    for m in moves:
        board[m] = HUMAN
        if winner(board) == HUMAN:
            board[m] = EMPTY
            return m
        board[m] = EMPTY

    # Heuristic 2: block an obvious opponent win
    for m in moves:
        board[m] = AI
        if winner(board) == AI:
            board[m] = EMPTY
            return m
        board[m] = EMPTY

    # Heuristic 3: prefer the center, then corners (common intuitive bias)
    for preferred in [4, 0, 2, 6, 8]:
        if preferred in moves:
            # 15% chance of an intuitive "slip" -> pick randomly instead
            if random.random() < 0.15:
                return random.choice(moves)
            return preferred

    return random.choice(moves)

# ---------------------------------------------------------------------
# Simulation
# ---------------------------------------------------------------------
def play_game(ai_starts=True):
    board = [EMPTY] * 9
    turn = AI if ai_starts else HUMAN
    ai_times, human_times = [], []

    while True:
        if turn == AI:
            t0 = time.perf_counter()
            move = ai_move(board)
            ai_times.append(time.perf_counter() - t0)
            board[move] = AI
            turn = HUMAN
        else:
            t0 = time.perf_counter()
            move = human_move(board)
            human_times.append(time.perf_counter() - t0)
            board[move] = HUMAN
            turn = AI

        result = winner(board)
        if result:
            return result, board, ai_times, human_times

def run_experiment(num_games=10):
    print("=" * 50)
    print("AI THINKING vs HUMAN THINKING - Tic-Tac-Toe Trial")
    print("=" * 50)

    tally = {"A": 0, "H": 0, "DRAW": 0}
    all_ai_times, all_human_times = [], []

    for g in range(num_games):
        ai_starts = (g % 2 == 0)
        result, board, ai_t, human_t = play_game(ai_starts)
        tally[result] += 1
        all_ai_times.extend(ai_t)
        all_human_times.extend(human_t)

        starter = "AI" if ai_starts else "Human"
        outcome = {"A": "AI wins", "H": "Human wins", "DRAW": "Draw"}[result]
        print(f"Game {g + 1:2d} | Starter: {starter:5s} | Result: {outcome}")

    print("\n" + "-" * 50)
    print("FINAL SCOREBOARD")
    print("-" * 50)
    print(f"AI wins : {tally['A']}")
    print(f"Human wins : {tally['H']}")
    print(f"Draws : {tally['DRAW']}")

    print("\n" + "-" * 50)
    print("DECISION-SPEED COMPARISON (avg time per move)")
    print("-" * 50)
    avg_ai = sum(all_ai_times) / len(all_ai_times)
    avg_human = sum(all_human_times) / len(all_human_times)
    print(f"AI (exhaustive Minimax search) : {avg_ai*1000:.4f} ms/move")
    print(f"Human (fast heuristic reasoning) : {avg_human*1000:.4f} ms/move")

    print("\n" + "-" * 50)
    print("INTERPRETATION")
    print("-" * 50)
    print("AI -> Slower per move, but logically exhaustive -> never loses.")
    print("Human -> Much faster, relies on pattern-recognition heuristics,")
    print(" occasionally sub-optimal, but computationally cheap.")
    print("=" * 50)

# Reproducible results, as stated in the report.
random.seed(42)
run_experiment(num_games=10)
